# AlistoCare Analysis Notebook
**Project:** AlistoCare: Health Center Queue Companion  
**Course:** BUSANALY – Business Analytics  
**Status:** Real data analysis (Kaggle Healthcare Analytics Patient Flow Data, CC0)  
**Date:** November 6, 2026

## ⚠️ Important Notice

This notebook runs on **real, anonymized patient encounter data** (`healthcare_analytics_cleaned.csv`).
All results are computed from actual patient records — no synthetic or fabricated records are included.

Complies with Data Privacy Act of 2012 (RA 10173): SSN-format IDs pseudonymized, name column dropped.


In [ ]:
# Setup
from pathlib import Path
import pandas as pd
import numpy as np

DATA_FILE = Path(__file__).parent / "healthcare_analytics_cleaned.csv" if '__file__' in dir() else Path.cwd() / "healthcare_analytics_cleaned.csv"
print(f"Looking for data at: {DATA_FILE.resolve()}")
print(f"File exists: {DATA_FILE.exists()}")

In [ ]:
# Load data
df = pd.read_csv(DATA_FILE)
print(f"Loaded {len(df):,} rows, {df.shape[1]} columns")
print(df.head())
print()
print(df.dtypes)

In [ ]:
# Structural validation
print("Rows:", len(df))
print("Columns:", df.shape[1])
print("Duplicate rows:", int(df.duplicated().sum()))
print("Duplicate patient_ids:", int(df["patient_id"].duplicated().sum()))
print("Missing values (total):", int(df.isna().sum().sum()))
print("Missing by column:")
print(df.isna().sum())

In [ ]:
# Compute derived fields
df["admission_datetime"] = pd.to_datetime(df["admission_datetime"], errors="coerce")
df["date"] = df["admission_datetime"].dt.date
df["day_of_week"] = df["admission_datetime"].dt.day_name()
df["hour"] = df["admission_datetime"].dt.hour
df["month"] = df["admission_datetime"].dt.month_name()
df["wait_category"] = pd.cut(df["wait_time_minutes"], bins=[0, 15, 30, 45, 60, float('inf')], labels=["0-15 min", "16-30 min", "31-45 min", "46-60 min", "60+ min"])
df["age_group"] = pd.cut(df["age"], bins=[0, 17, 30, 45, 60, 75, float('inf')], labels=["0-17", "18-30", "31-45", "46-60", "61-75", "76+"])

print(df[['date','day_of_week','hour','wait_time_minutes','wait_category','age_group']].head(10))

In [ ]:
# Descriptive analytics
total_encounters = len(df)
avg_wait = df["wait_time_minutes"].mean()
median_wait = df["wait_time_minutes"].median()
std_wait = df["wait_time_minutes"].std()
admission_rate = (df["admission_flag"] == "Admission").mean() * 100
avg_age = df["age"].mean()
sat_records = df["satisfaction_score"].notna().sum()
dept_records = df["department"].notna().sum()

print(f"Total patient encounters: {total_encounters:,}")
print(f"Average wait time: {avg_wait:.2f} minutes")
print(f"Median wait time: {median_wait:.2f} minutes")
print(f"Std deviation: {std_wait:.2f} minutes")
print(f"Average patient age: {avg_age:.1f} years")
print(f"Admission rate: {admission_rate:.1f}%")
print(f"Satisfaction records: {sat_records:,} ({sat_records/len(df)*100:.1f}%)")
print(f"Department records: {dept_records:,} ({dept_records/len(df)*100:.1f}%)")

In [ ]:
# Wait time by department
dept = df.groupby("department")["wait_time_minutes"].agg(["count", "mean", "median"]).round(2)
print(dept.sort_values("mean", ascending=False))

In [ ]:
# Wait time by day of week
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
dow = df.groupby("day_of_week")["wait_time_minutes"].agg(["count", "mean"]).round(2).reindex(day_order)
print(dow)

In [ ]:
# Wait time by hour
hour = df.groupby("hour")["wait_time_minutes"].agg(["count", "mean"]).round(2)
peak_hour = hour["mean"].idxmax()
low_hour = hour["mean"].idxmin()
print(f"Peak wait hour: {peak_hour}:00 ({hour.loc[peak_hour, 'mean']:.2f} min)")
print(f"Lowest wait hour: {low_hour}:00 ({hour.loc[low_hour, 'mean']:.2f} min)")
print(hour)

In [ ]:
# Diagnostic: admission flag comparison
adm = df.groupby("admission_flag")["wait_time_minutes"].agg(["count", "mean"]).round(2)
print(adm)
print()
# Satisfaction correlation
sat_df = df.dropna(subset=["satisfaction_score", "wait_time_minutes"])
corr = sat_df["wait_time_minutes"].corr(sat_df["satisfaction_score"])
print(f"Correlation wait time vs satisfaction: {corr:.4f}")
print(f"Satisfaction sample size: {len(sat_df):,}")

In [ ]:
# Predictive demonstration: wait time category classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

model_df = df.dropna(subset=["wait_category", "gender", "age", "department", "admission_flag", "hour"]).copy()
features = ["gender", "age", "department", "admission_flag", "hour"]
X = pd.get_dummies(model_df[features], drop_first=True)
y = model_df["wait_category"].astype(str)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(classification_report(y_test, y_pred, zero_division=0))
print("\nNOTE: REAL DATA — illustrative classification, not validated for production use.")

In [ ]:
# Prescriptive demonstration: capacity planning scenario
dept_waits = df.groupby("department")["wait_time_minutes"].mean().sort_values(ascending=False)
hour_waits = df.groupby("hour")["wait_time_minutes"].mean().sort_values(ascending=False)
long_waits = df[df["wait_time_minutes"] > 30]
long_waits_by_dept = long_waits.groupby("department").size()

print(f"Longest wait department: {dept_waits.idxmax()} ({dept_waits.max():.2f} min)")
print(f"Shortest wait department: {dept_waits.idxmin()} ({dept_waits.min():.2f} min)")
print(f"Peak wait hour: {hour_waits.idxmax()}:00 ({hour_waits.max():.2f} min)")
print(f"Patients waiting >30 min: {len(long_waits):,} ({(len(long_waits)/len(df)*100):.1f}%)")
print("\nLong waits by department:")
print(long_waits_by_dept.sort_values(ascending=False))
print("\nNOTE: REAL DATA — planning illustration based on observed patterns.")

In [ ]:
# Save cleaned data and summaries
output_dir = Path("analysis_output")
output_dir.mkdir(parents=True, exist_ok=True)

df.to_csv(output_dir / "alistocare_cleaned.csv", index=False)

summary = pd.DataFrame([
    {"metric": "total_encounters", "value": total_encounters},
    {"metric": "average_wait_time", "value": round(avg_wait, 2)},
    {"metric": "median_wait_time", "value": round(median_wait, 2)},
    {"metric": "std_wait_time", "value": round(std_wait, 2)},
    {"metric": "average_age", "value": round(avg_age, 1)},
    {"metric": "admission_rate", "value": round(admission_rate, 1)},
    {"metric": "satisfaction_records", "value": int(sat_records)},
    {"metric": "satisfaction_pct", "value": round(sat_records/len(df)*100, 1)},
    {"metric": "department_records", "value": int(dept_records)},
    {"metric": "department_pct", "value": round(dept_records/len(df)*100, 1)},
    {"metric": "wait_over_30_pct", "value": round((df["wait_time_minutes"] > 30).mean() * 100, 1)},
])
summary.to_csv(output_dir / "descriptive_summary.csv", index=False)

print(f"\nOutputs saved to: {output_dir.resolve()}")
print("\n✅ All outputs are from real, anonymized data.")

## Next Steps

1. All analysis is complete on real data — no synthetic records.
2. Outputs saved to `analysis_output/` folder.
3. Use results for dashboard, business report, and presentation.
4. All findings comply with RA 10173 (Data Privacy Act of 2012).